In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
def quality_gate(nome_tabela, df, schema_esperado, coluna_id_obrigatoria):
    """
    Função de Data Quality que verifica:
    1. Se existem IDs nulos.
    2. Se os tipos das colunas estão corretos.
    3. Se o schema é compatível.
    """
    print(f"Iniciando Quality Gate para: {nome_tabela}...")
    erros = []

    # --- CHECAGEM 1: ID NÃO NULO ---
    qtd_nulos = df.filter(F.col(coluna_id_obrigatoria).isNull()).count()
    if qtd_nulos > 0:
        erros.append(f"CRITICO: Encontrados {qtd_nulos} registros com '{coluna_id_obrigatoria}' NULO.")

    # --- CHECAGEM 2 e 3: TIPAGEM ---
    tipos_atuais = dict(df.dtypes)
    
    for col_nome, tipo_esperado in schema_esperado.items():
        if col_nome not in tipos_atuais:
            erros.append(f"SCHEMA: Coluna obrigatória '{col_nome}' desapareceu!")
        elif tipos_atuais[col_nome] != tipo_esperado:
            erros.append(f"TIPO: Coluna '{col_nome}' deveria ser '{tipo_esperado}', mas veio '{tipos_atuais[col_nome]}'.")

    # --- DECISÃO FINAL ---
    if erros:
        print(f"FALHA no Quality Gate de {nome_tabela}:")
        for e in erros: print(f"   - {e}")
        raise ValueError(f"Dados inválidos em {nome_tabela}. Processamento abortado.")
    else:
        print(f"{nome_tabela} passou em todos os testes de qualidade.")
        return True

In [0]:
# Notebook Orquestrador com Data Quality
print("Iniciando Pipeline Blindado - Case Rocket Lab...")

# Definição dos Schemas Esperados
# definindo o que NÃO pode mudar para garantir processamento futuro
schema_clientes = {
    "id_cliente": "bigint",
    "nome_cliente": "string",
    "idade": "int" 
}

schema_chamados = {
    "id_chamado": "int",
    "id_cliente": "bigint",
    "valor_custo": "decimal(18,6)",
    #-- "hora_abertura_chamado": "date"
}

try:
    # ---------------------------------------------------------
    # 1. Executa a Camada Silver (Ingestão)
    # ---------------------------------------------------------
    print("\nExecutando Camada Silver...")
    dbutils.notebook.run("/Users/contato.claraneves@gmail.com/RocketLab_VCredit_Case/notebooks/2_silver/transformacao_silver", 600)
    print("Silver concluída. Iniciando validação...")

    # ---------------------------------------------------------
    # 2. O Quality Gate (A Validação)
    # ---------------------------------------------------------
    catalogo = "medalhao_credit"
    silver_db_name = "silver_credit"
    
    # Carrega os dados recém-processados para checar
    df_cli = spark.table(f"{catalogo}.{silver_db_name}.ft_clientes") 
    df_cham = spark.table(f"{catalogo}.{silver_db_name}.ft_chamados_geral") 

    # Aplica as regras
    quality_gate("Tabela Clientes", df_cli, schema_clientes, "id_cliente")
    quality_gate("Tabela Chamados", df_cham, schema_chamados, "id_chamado")

    # ---------------------------------------------------------
    # 3. Executa a Camada Gold (Só roda se passar acima)
    # ---------------------------------------------------------
    print("\nValidação OK. Executando Camada Gold...")
    dbutils.notebook.run("./analise_gold", 600)
    
    print("\nPipeline finalizado com Sucesso e Qualidade Garantida!")

except Exception as e:
    print("\nPIPELINE ABORTADO!")
    print(f"Motivo: {e}")
    print("Nenhuma tabela Gold foi suja com dados ruins.")